In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import timedelta
import warnings
import os

warnings.filterwarnings('ignore')

# 한글 폰트 설정
import platform
system = platform.system()
if system == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif system == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)

# 경로 설정
DATA_PATH = Path('data')
PROCESSED_PATH = Path('data/processed')
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
# 파일 확인용. 작업 끝나면 지우기

if DATA_PATH.exists():
    data_files = []
    for file in sorted(DATA_PATH.iterdir()):
        if file.is_file() and file.suffix == '.csv':
            size_mb = file.stat().st_size / (1024 * 1024)
            data_files.append({
                'File': file.name,
                'Size (MB)': f"{size_mb:.1f}",
                'Size_numeric': size_mb
            })
    
    df_files = pd.DataFrame(data_files)
    df_files = df_files.sort_values('Size_numeric', ascending=False)
    
    print("\n사용 가능한 CSV 파일:")
    for idx, row in df_files.iterrows():
        status = "✓" if row['Size_numeric'] > 0 else "✗"
        print(f"  {status} {row['File']:<30} {row['Size (MB)']:>10} MB")
    
    print(f"\n총 {len(df_files)}개 파일")
    print(f"총 크기: {df_files['Size_numeric'].sum():.1f} MB")
else:
    print("⚠ 데이터 디렉토리를 찾을 수 없습니다!")

In [ ]:
# 데이터 백업 함수
def save_checkpoint(df, part_name):
    """체크포인트 저장 헬퍼 함수"""
    save_file = PROCESSED_PATH / f'features_after_{part_name}.csv'
    df.to_csv(save_file, index=False)
    
    columns_file = PROCESSED_PATH / f'columns_after_{part_name}.txt'
    with open(columns_file, 'w') as f:
        f.write('\n'.join(df.columns))
    
    size_mb = save_file.stat().st_size / (1024*1024)
    
    print(f"\n{'='*80}")
    print(f"✓ 체크포인트 저장: {part_name.upper()}")
    print(f"{'='*80}")
    print(f"  파일: {save_file.name}")
    print(f"  크기: {size_mb:.1f} MB")
    print(f"  행: {len(df):,}건")
    print(f"  열: {len(df.columns)}개")
    
# 사용법
# save_checkpoint(df_features, 'part2')

**데이터 로드**

In [ ]:
# 최종 코호트 로드
cohort_file = PROCESSED_PATH / 'final_cohort.csv'

if cohort_file.exists():
    df_cohort = pd.read_csv(cohort_file)
    
    # 날짜 컬럼 파싱
    date_cols = ['admittime', 'dischtime']
    for col in date_cols:
        if col in df_cohort.columns:
            df_cohort[col] = pd.to_datetime(df_cohort[col])
    
    print(f"✓ 코호트 로드 완료: {len(df_cohort):,}건")
    print(f"  - 재입원: {df_cohort['readmission_30d'].sum():,}건 ({df_cohort['readmission_30d'].mean()*100:.2f}%)")
    print(f"  - 고유 환자: {df_cohort['subject_id'].nunique():,}명")
    print(f"  - 컬럼: {list(df_cohort.columns)}")
    
    # 코호트의 hadm_id 집합 (필터링용)
    cohort_hadm_ids = set(df_cohort['hadm_id'].unique())
    cohort_subject_ids = set(df_cohort['subject_id'].unique())
    
    print(f"\n코호트 정보:")
    print(f"  - hadm_id 수: {len(cohort_hadm_ids):,}개")
    print(f"  - subject_id 수: {len(cohort_subject_ids):,}개")
    
    # 샘플 확인
    print(f"\n샘플 데이터:")
    print(df_cohort.head(3))
    
else:
    print("⚠ 최종 코호트 파일을 찾을 수 없습니다!")
    print(f"  경로: {cohort_file}")
    print("  01_cohort_extraction.ipynb를 먼저 실행해주세요.")

In [ ]:
# 크기 작은 데이터 로드 (parients, admissions, icustays, d_icd_diagnoses)
# Patients (인구학적 정보)
print("\n[1] Patients 로드...")
df_patients = pd.read_csv(DATA_PATH / 'patients.csv')
print(f"✓ Patients: {len(df_patients):,}명")

# Admissions (입원 정보)
print("\n[2] Admissions 로드...")
df_admissions = pd.read_csv(DATA_PATH / 'admissions.csv')
date_cols = ['admittime', 'dischtime', 'deathtime', 'edregtime', 'edouttime']
for col in date_cols:
    if col in df_admissions.columns:
        df_admissions[col] = pd.to_datetime(df_admissions[col])
print(f"✓ Admissions: {len(df_admissions):,}건")

# ICU Stays
print("\n[3] ICU Stays 로드...")
df_icustays = pd.read_csv(DATA_PATH / 'icustays.csv')
date_cols_icu = ['intime', 'outtime']
for col in date_cols_icu:
    if col in df_icustays.columns:
        df_icustays[col] = pd.to_datetime(df_icustays[col])
print(f"✓ ICU Stays: {len(df_icustays):,}건")

# D_ICD_Diagnoses (진단 코드 사전)
print("\n[4] ICD Diagnoses Dictionary 로드...")
df_d_icd = pd.read_csv(DATA_PATH / 'd_icd_diagnoses.csv')
print(f"✓ ICD Dictionary: {len(df_d_icd):,}개 코드")

print("\n✓ 기본 데이터 로드 완료")

In [ ]:
#메모리 사용량 확인
import sys

def get_size(df, name):
    size_mb = sys.getsizeof(df) / (1024 * 1024)
    return f"{name}: {size_mb:.1f} MB"

print(f"\n현재 메모리 사용:")
print(f"  {get_size(df_cohort, 'df_cohort')}")
print(f"  {get_size(df_patients, 'df_patients')}")
print(f"  {get_size(df_admissions, 'df_admissions')}")
print(f"  {get_size(df_icustays, 'df_icustays')}")
print(f"  {get_size(df_d_icd, 'df_d_icd')}")

**피처 추출**

patients table - 인구학적 피처 추출

In [ ]:
# patients table 피처 추출
# 코호트와 환자 정보 병합
df_features = df_cohort.copy()

# Patients 테이블에서 필요한 정보 추출
df_patients_info = df_patients[['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']]

# 병합
df_features = df_features.merge(
    df_patients_info,
    on='subject_id',
    how='left'
)

print(f"✓ 환자 정보 병합 완료: {len(df_features):,}건")

# 사망일 파싱
if 'dod' in df_features.columns:
    df_features['dod'] = pd.to_datetime(df_features['dod'])

# 기본 통계
print(f"\n기본 인구학적 통계:")
print(f"  - 평균 연령: {df_features['anchor_age'].mean():.1f}세")
print(f"  - 연령 범위: {df_features['anchor_age'].min()}-{df_features['anchor_age'].max()}세")
print(f"  - 성별 분포:")
print(df_features['gender'].value_counts())

In [ ]:
# 나이 피처 생성
# 입원 시점 나이 계산
# anchor_year에서 anchor_age였고, 실제 입원은 admittime
# admittime의 년도와 anchor_year의 차이를 anchor_age에 더함

df_features['admit_year'] = df_features['admittime'].dt.year
df_features['age_at_admission'] = df_features['anchor_age'] + (
    df_features['admit_year'] - df_features['anchor_year']
)

# 나이 범주화
df_features['age_group'] = pd.cut(
    df_features['age_at_admission'],
    bins=[0, 40, 50, 60, 70, 80, 200],
    labels=['<40', '40-49', '50-59', '60-69', '70-79', '≥80']
)

# 나이 관련 피처
df_features['age_over_65'] = (df_features['age_at_admission'] >= 65).astype(int)
df_features['age_over_75'] = (df_features['age_at_admission'] >= 75).astype(int)
df_features['age_over_85'] = (df_features['age_at_admission'] >= 85).astype(int)

print(f"✓ 나이 피처 생성 완료")
print(f"\n나이 분포:")
print(df_features['age_group'].value_counts().sort_index())
print(f"\n65세 이상: {df_features['age_over_65'].sum():,}명 ({df_features['age_over_65'].mean()*100:.1f}%)")
print(f"75세 이상: {df_features['age_over_75'].sum():,}명 ({df_features['age_over_75'].mean()*100:.1f}%)")
print(f"85세 이상: {df_features['age_over_85'].sum():,}명 ({df_features['age_over_85'].mean()*100:.1f}%)")

In [ ]:
# 성별 피처 생성
# 원-핫 인코딩 사용
# 성별을 숫자로 변환 (M=1, F=0)
df_features['gender_male'] = (df_features['gender'] == 'M').astype(int)
df_features['gender_female'] = (df_features['gender'] == 'F').astype(int)

print(f"✓ 성별 피처 생성 완료")
print(f"  - 남성: {df_features['gender_male'].sum():,}명 ({df_features['gender_male'].mean()*100:.1f}%)")
print(f"  - 여성: {df_features['gender_female'].sum():,}명 ({df_features['gender_female'].mean()*100:.1f}%)")

# 재입원율 성별 비교
print(f"\n성별별 재입원율:")
readmit_by_gender = df_features.groupby('gender')['readmission_30d'].agg(['sum', 'count', 'mean'])
readmit_by_gender['percentage'] = readmit_by_gender['mean'] * 100
print(readmit_by_gender)

In [ ]:
# 입원 유형 및 시간 관련 피처
# 입원 유형 (이미 코호트에 있지만 확인)
if 'admission_type' in df_features.columns:
    print("입원 유형 분포:")
    print(df_features['admission_type'].value_counts().head(10))
    
    # 주요 입원 유형 원-핫 인코딩
    df_features['admit_emergency'] = df_features['admission_type'].str.contains(
        'EMER|URGENT', case=False, na=False
    ).astype(int)
    
    df_features['admit_elective'] = df_features['admission_type'].str.contains(
        'ELECTIVE', case=False, na=False
    ).astype(int)
    
    df_features['admit_observation'] = df_features['admission_type'].str.contains(
        'OBSERVATION', case=False, na=False
    ).astype(int)
    
    print(f"\n✓ 입원 유형 피처:")
    print(f"  - 응급/긴급: {df_features['admit_emergency'].sum():,}건 ({df_features['admit_emergency'].mean()*100:.1f}%)")
    print(f"  - 선택적: {df_features['admit_elective'].sum():,}건 ({df_features['admit_elective'].mean()*100:.1f}%)")
    print(f"  - 관찰: {df_features['admit_observation'].sum():,}건 ({df_features['admit_observation'].mean()*100:.1f}%)")

# 입원 시간 관련 피처
df_features['admit_hour'] = df_features['admittime'].dt.hour
df_features['admit_day_of_week'] = df_features['admittime'].dt.dayofweek  # 0=월요일
df_features['admit_month'] = df_features['admittime'].dt.month
df_features['admit_weekend'] = (df_features['admit_day_of_week'] >= 5).astype(int)  # 토, 일

print(f"\n시간 관련 피처:")
print(f"  - 주말 입원: {df_features['admit_weekend'].sum():,}건 ({df_features['admit_weekend'].mean()*100:.1f}%)")

In [ ]:
# 입원 기간 (LOS) 관련 피처
# LOS는 이미 코호트에 있음
if 'los_days' in df_features.columns:
    print(f"입원 기간 통계:")
    print(df_features['los_days'].describe())
    
    # LOS 범주화
    df_features['los_short'] = (df_features['los_days'] < 3).astype(int)  # 3일 미만
    df_features['los_normal'] = ((df_features['los_days'] >= 3) & (df_features['los_days'] < 7)).astype(int)
    df_features['los_long'] = (df_features['los_days'] >= 7).astype(int)  # 7일 이상
    df_features['los_very_long'] = (df_features['los_days'] >= 14).astype(int)  # 14일 이상
    
    print(f"\n입원 기간 분포:")
    print(f"  - 단기 (<3일): {df_features['los_short'].sum():,}건 ({df_features['los_short'].mean()*100:.1f}%)")
    print(f"  - 일반 (3-7일): {df_features['los_normal'].sum():,}건 ({df_features['los_normal'].mean()*100:.1f}%)")
    print(f"  - 장기 (≥7일): {df_features['los_long'].sum():,}건 ({df_features['los_long'].mean()*100:.1f}%)")
    print(f"  - 매우 장기 (≥14일): {df_features['los_very_long'].sum():,}건 ({df_features['los_very_long'].mean()*100:.1f}%)")
    
    # 재입원율 LOS별 비교
    print(f"\nLOS별 재입원율:")
    for los_type, col in [('단기', 'los_short'), ('일반', 'los_normal'), ('장기', 'los_long')]:
        mask = df_features[col] == 1
        if mask.sum() > 0:
            readmit_rate = df_features[mask]['readmission_30d'].mean() * 100
            print(f"  - {los_type}: {readmit_rate:.2f}%")

In [ ]:
# 인구학적 피처 요약
# 생성된 피처 목록
demographic_features = [
    'age_at_admission', 'age_group', 'age_over_65', 'age_over_75', 'age_over_85',
    'gender_male', 'gender_female',
    'admit_emergency', 'admit_elective', 'admit_observation',
    'admit_hour', 'admit_day_of_week', 'admit_month', 'admit_weekend',
    'los_short', 'los_normal', 'los_long', 'los_very_long'
]

# 실제 존재하는 피처만 필터링
existing_features = [f for f in demographic_features if f in df_features.columns]

print(f"\n생성된 피처 수: {len(existing_features)}개")
print(f"피처 목록:")
for i, feat in enumerate(existing_features, 1):
    print(f"  {i:2d}. {feat}")

print(f"\n현재 데이터프레임 크기:")
print(f"  - 행: {len(df_features):,}건")
print(f"  - 열: {len(df_features.columns)}개")

# 결측치 확인
print(f"\n결측치 확인 (새로 생성된 피처):")
for feat in existing_features:
    missing = df_features[feat].isnull().sum()
    if missing > 0:
        print(f"  ⚠ {feat}: {missing:,}건 ({missing/len(df_features)*100:.2f}%)")

In [ ]:
save_checkpoint(df_features, 'part2')

diagnosis table - 진단 피처 추출

In [ ]:
# Diagnoses ICD 로드 (청크 단위로 - 메모리 효율)
# 정의한 코호트에 해당하는 진단만 필터링
df_diagnoses = pd.read_csv(DATA_PATH / 'diagnoses_icd.csv')

print(f"✓ 전체 진단: {len(df_diagnoses):,}건")

# 코호트에 속한 입원만 필터링
df_diagnoses_cohort = df_diagnoses[df_diagnoses['hadm_id'].isin(cohort_hadm_ids)].copy()

print(f"✓ 코호트 진단: {len(df_diagnoses_cohort):,}건")
print(f"  - 고유 hadm_id: {df_diagnoses_cohort['hadm_id'].nunique():,}개")
print(f"  - 고유 ICD 코드: {df_diagnoses_cohort['icd_code'].nunique():,}개")

# 메모리 절약
del df_diagnoses
import gc
gc.collect()

print(f"\n샘플 데이터:")
print(df_diagnoses_cohort.head())

In [ ]:
# 주요 동반질환 ICD 코드 정의
# 심부전 재입원과 관련된 주요 동반질환
# Elixhauser Comorbidity Index + 심부전 특이적 질환

comorbidities = {
    # 심혈관계
    'hypertension': {
        'name': '고혈압',
        'icd10': ['I10', 'I11', 'I12', 'I13', 'I15'],
        'icd9': ['401', '402', '403', '404', '405']
    },
    'coronary_artery_disease': {
        'name': '관상동맥질환',
        'icd10': ['I20', 'I21', 'I22', 'I23', 'I24', 'I25'],
        'icd9': ['410', '411', '412', '413', '414']
    },
    'atrial_fibrillation': {
        'name': '심방세동',
        'icd10': ['I48'],
        'icd9': ['4273']
    },
    'valvular_disease': {
        'name': '판막질환',
        'icd10': ['I05', 'I06', 'I07', 'I08', 'I34', 'I35', 'I36', 'I37'],
        'icd9': ['394', '395', '396', '397', '424']
    },
    
    # 신장/대사
    'chronic_kidney_disease': {
        'name': '만성신장질환',
        'icd10': ['N18', 'N19'],
        'icd9': ['585', '586']
    },
    'diabetes': {
        'name': '당뇨병',
        'icd10': ['E10', 'E11', 'E13', 'E14'],
        'icd9': ['250']
    },
    'diabetes_with_complications': {
        'name': '합병증 동반 당뇨',
        'icd10': ['E102', 'E103', 'E104', 'E105', 'E112', 'E113', 'E114', 'E115'],
        'icd9': ['2504', '2505', '2506', '2507']
    },
    
    # 호흡기계
    'copd': {
        'name': '만성폐쇄성폐질환',
        'icd10': ['J44'],
        'icd9': ['490', '491', '492', '496']
    },
    'pneumonia': {
        'name': '폐렴',
        'icd10': ['J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18'],
        'icd9': ['480', '481', '482', '483', '484', '485', '486']
    },
    
    # 간/소화기
    'liver_disease': {
        'name': '간질환',
        'icd10': ['K70', 'K71', 'K72', 'K73', 'K74', 'K76'],
        'icd9': ['570', '571', '572', '573']
    },
    
    # 혈액
    'anemia': {
        'name': '빈혈',
        'icd10': ['D50', 'D51', 'D52', 'D53', 'D63', 'D64'],
        'icd9': ['280', '281', '285']
    },
    
    # 신경/정신
    'dementia': {
        'name': '치매',
        'icd10': ['F00', 'F01', 'F02', 'F03', 'G30'],
        'icd9': ['290', '294', '3312']
    },
    'depression': {
        'name': '우울증',
        'icd10': ['F32', 'F33'],
        'icd9': ['296', '311']
    },
    
    # 기타
    'obesity': {
        'name': '비만',
        'icd10': ['E66'],
        'icd9': ['2780']
    },
    'cancer': {
        'name': '암',
        'icd10': ['C'],  # C로 시작하는 모든 코드
        'icd9': ['1', '2']  # 140-239
    }
}

print(f"✓ 정의된 동반질환: {len(comorbidities)}개")
for key, value in comorbidities.items():
    print(f"  - {value['name']} ({key})")

In [ ]:
# 동반질환 피처 생성
# 각 hadm_id별로 동반질환 확인
comorbidity_features = {}

for hadm_id in cohort_hadm_ids:
    comorbidity_features[hadm_id] = {}

# 진단 코드 병합 (ICD 사전과)
df_diagnoses_cohort = df_diagnoses_cohort.merge(
    df_d_icd[['icd_code', 'icd_version', 'long_title']],
    on=['icd_code', 'icd_version'],
    how='left'
)

for comorb_key, comorb_info in comorbidities.items():
    print(f"  처리 중: {comorb_info['name']}...", end='')
    
    # ICD-10 매칭
    mask_icd10 = df_diagnoses_cohort['icd_version'] == 10
    mask_codes_10 = df_diagnoses_cohort['icd_code'].str.startswith(
        tuple(comorb_info['icd10']), na=False
    )
    
    # ICD-9 매칭
    mask_icd9 = df_diagnoses_cohort['icd_version'] == 9
    if comorb_key == 'cancer':
        # 암은 특별 처리 (140-239)
        mask_codes_9 = (
            df_diagnoses_cohort['icd_code'].str.startswith('1', na=False) |
            df_diagnoses_cohort['icd_code'].str.startswith('2', na=False)
        )
    else:
        mask_codes_9 = df_diagnoses_cohort['icd_code'].str.startswith(
            tuple(comorb_info['icd9']), na=False
        )
    
    # 해당 동반질환이 있는 hadm_id
    mask_total = (mask_icd10 & mask_codes_10) | (mask_icd9 & mask_codes_9)
    hadm_with_comorb = set(df_diagnoses_cohort[mask_total]['hadm_id'].unique())
    
    # 각 hadm_id에 피처 할당
    for hadm_id in cohort_hadm_ids:
        comorbidity_features[hadm_id][f'comorb_{comorb_key}'] = int(hadm_id in hadm_with_comorb)
    
    print(f" {len(hadm_with_comorb):,}건")

print("\n✓ 동반질환 피처 생성 완료")

In [ ]:
# 동반질환 데이터프레임 생성")
# 딕셔너리를 데이터프레임으로 변환
df_comorbidities = pd.DataFrame.from_dict(comorbidity_features, orient='index')
df_comorbidities.reset_index(inplace=True)
df_comorbidities.rename(columns={'index': 'hadm_id'}, inplace=True)

print(f"✓ 동반질환 데이터프레임 생성: {len(df_comorbidities):,}건")
print(f"  - 컬럼 수: {len(df_comorbidities.columns)}개")

# 메인 피처 데이터프레임과 병합
df_features = df_features.merge(
    df_comorbidities,
    on='hadm_id',
    how='left'
)

print(f"\n✓ 병합 완료")
print(f"  - 최종 행 수: {len(df_features):,}건")
print(f"  - 최종 열 수: {len(df_features.columns)}개")

# 동반질환 통계
print("\n동반질환 유병률:")
for comorb_key, comorb_info in comorbidities.items():
    col_name = f'comorb_{comorb_key}'
    if col_name in df_features.columns:
        count = df_features[col_name].sum()
        pct = count / len(df_features) * 100
        print(f"  - {comorb_info['name']:<20}: {count:>6,}건 ({pct:>5.1f}%)")

In [ ]:
# 동반질환 지수 계산
# 동반질환 개수 (단순 합산)
comorbidity_cols = [col for col in df_features.columns if col.startswith('comorb_')]
df_features['comorbidity_count'] = df_features[comorbidity_cols].sum(axis=1)

print(f"✓ 동반질환 개수 계산 완료")
print(f"\n동반질환 개수 분포:")
print(df_features['comorbidity_count'].value_counts().sort_index().head(10))

# 동반질환 범주화
df_features['comorbidity_none'] = (df_features['comorbidity_count'] == 0).astype(int)
df_features['comorbidity_low'] = ((df_features['comorbidity_count'] >= 1) & 
                                   (df_features['comorbidity_count'] <= 2)).astype(int)
df_features['comorbidity_moderate'] = ((df_features['comorbidity_count'] >= 3) & 
                                        (df_features['comorbidity_count'] <= 5)).astype(int)
df_features['comorbidity_high'] = (df_features['comorbidity_count'] > 5).astype(int)

print(f"\n동반질환 부담 분포:")
print(f"  - 없음 (0): {df_features['comorbidity_none'].sum():,}건 ({df_features['comorbidity_none'].mean()*100:.1f}%)")
print(f"  - 낮음 (1-2): {df_features['comorbidity_low'].sum():,}건 ({df_features['comorbidity_low'].mean()*100:.1f}%)")
print(f"  - 중등도 (3-5): {df_features['comorbidity_moderate'].sum():,}건 ({df_features['comorbidity_moderate'].mean()*100:.1f}%)")
print(f"  - 높음 (>5): {df_features['comorbidity_high'].sum():,}건 ({df_features['comorbidity_high'].mean()*100:.1f}%)")

# 재입원율과의 관계
print(f"\n동반질환 부담별 재입원율:")
for burden_type, col in [('없음', 'comorbidity_none'), 
                         ('낮음', 'comorbidity_low'), 
                         ('중등도', 'comorbidity_moderate'), 
                         ('높음', 'comorbidity_high')]:
    mask = df_features[col] == 1
    if mask.sum() > 0:
        readmit_rate = df_features[mask]['readmission_30d'].mean() * 100
        n = mask.sum()
        print(f"  - {burden_type}: {readmit_rate:.2f}% (n={n:,})")

In [ ]:
# diagnosis table진단 피처 요약
# 생성된 피처 목록
diagnosis_features = comorbidity_cols + [
    'comorbidity_count',
    'comorbidity_none',
    'comorbidity_low',
    'comorbidity_moderate',
    'comorbidity_high'
]

print(f"\n생성된 진단 피처 수: {len(diagnosis_features)}개")
print(f"  - 개별 동반질환: {len(comorbidity_cols)}개")
print(f"  - 동반질환 지수: 5개")

print(f"\n현재 데이터프레임 크기:")
print(f"  - 행: {len(df_features):,}건")
print(f"  - 열: {len(df_features.columns)}개")

# 메모리 정리
#del df_diagnoses_cohort
#del comorbidity_features
#gc.collect()

In [ ]:
save_checkpoint(df_features, 'part3')

**prescription table**

In [ ]:
# 현재 메모리 상태 확인
import psutil
import os

process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / 1024 / 1024  # MB
print(f"현재 메모리 사용량: {mem_before:.1f} MB")

In [ ]:
# 심부전 치료 약물 정의
hf_medications = {
    # 1. 이뇨제 (Loop/Thiazide만)
    'diuretics': {
        'name': '이뇨제',
        'drugs': ['furosemide', 'lasix', 'bumetanide', 'bumex', 'torsemide', 'demadex',
                  'metolazone', 'zaroxolyn', 'hydrochlorothiazide', 'hctz',
                  'chlorthalidone']
    },
    
    # 2. 베타 차단제
    'beta_blockers': {
        'name': '베타차단제',
        'drugs': ['metoprolol', 'lopressor', 'toprol', 'carvedilol', 'coreg',
                  'bisoprolol', 'zebeta', 'atenolol', 'tenormin',
                  'propranolol', 'inderal', 'labetalol', 'trandate']
    },
    
    # 3. ACE 억제제
    'ace_inhibitors': {
        'name': 'ACE억제제',
        'drugs': ['lisinopril', 'prinivil', 'zestril', 'enalapril', 'vasotec',
                  'captopril', 'capoten', 'ramipril', 'altace',
                  'quinapril', 'accupril', 'benazepril', 'lotensin',
                  'fosinopril', 'monopril', 'perindopril', 'aceon']
    },
    
    # 4. ARB
    'arbs': {
        'name': 'ARB',
        'drugs': ['losartan', 'cozaar', 'candesartan', 'atacand', 
                  'irbesartan', 'avapro', 'telmisartan', 'micardis', 
                  'olmesartan', 'benicar']
    },
    
    # 5. MRA (알도스테론 길항제)
    'mra': {
        'name': '알도스테론길항제',
        'drugs': ['spironolactone', 'aldactone', 'eplerenone', 'inspra']
    },
    
    # 6. ARNI
    'arni': {
        'name': 'ARNI',
        'drugs': ['sacubitril', 'entresto']
    },
    
    # 7. 디곡신
    'digoxin': {
        'name': '디곡신',
        'drugs': ['digoxin', 'lanoxin']
    },
    
    # 8. 질산염
    'nitrates': {
        'name': '질산염',
        'drugs': ['nitroglycerin', 'nitro', 'isosorbide', 'imdur', 'isordil',
                  'hydralazine']
    },
    
    # 9. 칼슘채널차단제
    'calcium_blockers': {
        'name': '칼슘채널차단제',
        'drugs': ['amlodipine', 'norvasc', 'diltiazem', 'cardizem',
                  'verapamil', 'calan', 'nifedipine', 'procardia']
    },
    
    # 10. 항응고제
    'anticoagulants': {
        'name': '항응고제',
        'drugs': ['warfarin', 'coumadin', 'heparin', 'enoxaparin', 'lovenox',
                  'apixaban', 'eliquis', 'rivaroxaban', 'xarelto',
                  'dabigatran', 'pradaxa', 'edoxaban', 'savaysa']
    },
    
    # 11. 스타틴
    'statins': {
        'name': '스타틴',
        'drugs': ['atorvastatin', 'lipitor', 'simvastatin', 'zocor',
                  'rosuvastatin', 'crestor', 'pravastatin', 'pravachol',
                  'lovastatin', 'mevacor', 'fluvastatin', 'lescol']
    },
    
    # 12. 항부정맥제
    'antiarrhythmics': {
        'name': '항부정맥제',
        'drugs': ['amiodarone', 'cordarone', 'pacerone', 'sotalol', 'betapace',
                  'dofetilide', 'tikosyn', 'dronedarone', 'multaq']
    }
}

print(f"✓ 정의된 약물 클래스: {len(hf_medications)}개")
for key, value in hf_medications.items():
    print(f"  - {value['name']} ({key}): {len(value['drugs'])}개 약물")

all_drug_keywords = set()
for med_class in hf_medications.values():
    all_drug_keywords.update([drug.lower() for drug in med_class['drugs']])

print(f"\n✓ 총 약물 키워드: {len(all_drug_keywords)}개")

# 중복 확인
from collections import defaultdict
drug_to_classes = defaultdict(list)
for med_class, med_info in hf_medications.items():
    for drug in med_info['drugs']:
        drug_to_classes[drug.lower()].append(med_class)

duplicates = {drug: classes for drug, classes in drug_to_classes.items() if len(classes) > 1}
if duplicates:
    print(f"\n⚠ 중복된 약물: {len(duplicates)}개")
    for drug, classes in duplicates.items():
        print(f"  '{drug}': {classes}")
else:
    print(f"\n✓ 중복 없음!")

In [ ]:
# Prescriptions 데이터 청크 단위 처리
# 약물 피처를 저장할 딕셔너리
medication_features = {hadm_id: {} for hadm_id in cohort_hadm_ids}

# 초기화
for hadm_id in cohort_hadm_ids:
    for med_key in hf_medications.keys():
        medication_features[hadm_id][f'med_{med_key}'] = 0

# 청크 크기 설정 (메모리에 따라 조정 가능)
chunk_size = 500000  # 한 번에 50만 행씩

print(f"청크 크기: {chunk_size:,}행")
print("\n처리 시작...")

chunks_processed = 0
total_prescriptions = 0
matched_prescriptions = 0

# 청크 단위로 읽기
for chunk in pd.read_csv(DATA_PATH / 'prescriptions.csv', chunksize=chunk_size):
    chunks_processed += 1
    total_prescriptions += len(chunk)
    
    # 우리 코호트에 해당하는 처방만 필터링
    chunk_filtered = chunk[chunk['hadm_id'].isin(cohort_hadm_ids)].copy()
    matched_prescriptions += len(chunk_filtered)
    
    if len(chunk_filtered) == 0:
        print(f"  청크 {chunks_processed}: {len(chunk):,}행 처리 (매칭 0건)", end='\r')
        continue
    
    # 약물명을 소문자로 변환
    chunk_filtered['drug_lower'] = chunk_filtered['drug'].str.lower()
    
    # 각 약물 클래스별로 매칭
    for med_key, med_info in hf_medications.items():
        # 약물명 매칭 (부분 문자열 포함)
        for drug_keyword in med_info['drugs']:
            mask = chunk_filtered['drug_lower'].str.contains(drug_keyword, na=False, case=False)
            matched_hadm_ids = chunk_filtered[mask]['hadm_id'].unique()
            
            # 해당 hadm_id에 약물 표시
            for hadm_id in matched_hadm_ids:
                medication_features[hadm_id][f'med_{med_key}'] = 1
    
    print(f"  청크 {chunks_processed}: {len(chunk):,}행 처리 (매칭 {len(chunk_filtered):,}건)", end='\r')

print(f"\n\n✓ 처리 완료!")
print(f"  - 총 청크 수: {chunks_processed}개")
print(f"  - 총 처방 수: {total_prescriptions:,}건")
print(f"  - 코호트 매칭: {matched_prescriptions:,}건")

# 메모리 확인
mem_after = process.memory_info().rss / 1024 / 1024  # MB
print(f"\n메모리 사용량:")
print(f"  - 이전: {mem_before:.1f} MB")
print(f"  - 이후: {mem_after:.1f} MB")
print(f"  - 증가: {mem_after - mem_before:.1f} MB")

In [ ]:
# 약물 피처 데이터프레임 생성")
# 딕셔너리를 데이터프레임으로 변환
df_medications = pd.DataFrame.from_dict(medication_features, orient='index')
df_medications.reset_index(inplace=True)
df_medications.rename(columns={'index': 'hadm_id'}, inplace=True)

print(f"✓ 약물 데이터프레임 생성: {len(df_medications):,}건")
print(f"  - 컬럼 수: {len(df_medications.columns)}개")

# 메인 피처 데이터프레임과 병합
df_features = df_features.merge(
    df_medications,
    on='hadm_id',
    how='left'
)

print(f"\n✓ 병합 완료")
print(f"  - 최종 행 수: {len(df_features):,}건")
print(f"  - 최종 열 수: {len(df_features.columns)}개")

# 약물 처방률
print("\n약물 클래스별 처방률:")
for med_key, med_info in hf_medications.items():
    col_name = f'med_{med_key}'
    if col_name in df_features.columns:
        count = df_features[col_name].sum()
        pct = count / len(df_features) * 100
        print(f"  - {med_info['name']:<20}: {count:>6,}건 ({pct:>5.1f}%)")

In [ ]:
# 가이드라인 준수도 및 약물 개수 계산
# 이 코호트 시기(2008-2019)의 표준: Triple Therapy
# RAAS 억제 + 베타차단제 + MRA
print("이 코호트 시기의 GDMT 표준:")
print("  Triple Therapy (3제):")
print("  1. ARNI/ACE억제제/ARB (RAAS 억제)")
print("  2. 베타차단제")
print("  3. 미네랄코르티코이드 수용체 길항제 (MRA)")
print()
print("⚠ 참고: 현재 표준(2024)은 Quadruple Therapy (+ SGLT2i)")
print("   하지만 이 데이터(2008-2019)에는 SGLT2i가 없음\n")

# 1. ARNI/ACE/ARB
df_features['gdmt_raas'] = (
    (df_features['med_arni'] == 1) |
    (df_features['med_ace_inhibitors'] == 1) | 
    (df_features['med_arbs'] == 1)
).astype(int)

# 2. 베타차단제
df_features['gdmt_bb'] = df_features['med_beta_blockers']

# 3. MRA
df_features['gdmt_mra'] = df_features['med_mra']

# GDMT 준수도
# Triple Therapy (3제 모두)
df_features['gdmt_triple'] = (
    (df_features['gdmt_raas'] == 1) &
    (df_features['gdmt_bb'] == 1) &
    (df_features['gdmt_mra'] == 1)
).astype(int)

# Dual Therapy (RAAS + BB)
df_features['gdmt_dual'] = (
    (df_features['gdmt_raas'] == 1) &
    (df_features['gdmt_bb'] == 1)
).astype(int)

# Essential Diuretics
df_features['on_diuretics'] = df_features['med_diuretics']

# 약물 개수
medication_cols = [col for col in df_features.columns if col.startswith('med_')]
df_features['medication_count'] = df_features[medication_cols].sum(axis=1)

# 약물 개수 범주화
df_features['medication_none'] = (df_features['medication_count'] == 0).astype(int)
df_features['medication_low'] = ((df_features['medication_count'] >= 1) & 
                                  (df_features['medication_count'] <= 3)).astype(int)
df_features['medication_moderate'] = ((df_features['medication_count'] >= 4) & 
                                       (df_features['medication_count'] <= 6)).astype(int)
df_features['medication_high'] = (df_features['medication_count'] > 6).astype(int)

# 다약제
df_features['polypharmacy'] = (df_features['medication_count'] >= 5).astype(int)

# 결과 출력
print("=" * 80)
print("GDMT 준수도 (Triple Therapy 기준):")
print("-" * 80)
print(f"  RAAS 억제: {df_features['gdmt_raas'].sum():,}건 ({df_features['gdmt_raas'].mean()*100:.1f}%)")
print(f"  베타차단제: {df_features['gdmt_bb'].sum():,}건 ({df_features['gdmt_bb'].mean()*100:.1f}%)")
print(f"  MRA: {df_features['gdmt_mra'].sum():,}건 ({df_features['gdmt_mra'].mean()*100:.1f}%)")

print(f"\nGDMT 조합:")
print(f"  - Triple Therapy (3제): {df_features['gdmt_triple'].sum():,}건 ({df_features['gdmt_triple'].mean()*100:.1f}%)")
print(f"  - Dual Therapy (2제): {df_features['gdmt_dual'].sum():,}건 ({df_features['gdmt_dual'].mean()*100:.1f}%)")
print(f"  - 이뇨제 처방: {df_features['on_diuretics'].sum():,}건 ({df_features['on_diuretics'].mean()*100:.1f}%)")

print(f"\n약물 개수 분포:")
print(df_features['medication_count'].value_counts().sort_index().head(15))

print(f"\n약물 부담 분포:")
print(f"  - 없음 (0): {df_features['medication_none'].sum():,}건 ({df_features['medication_none'].mean()*100:.1f}%)")
print(f"  - 낮음 (1-3): {df_features['medication_low'].sum():,}건 ({df_features['medication_low'].mean()*100:.1f}%)")
print(f"  - 중등도 (4-6): {df_features['medication_moderate'].sum():,}건 ({df_features['medication_moderate'].mean()*100:.1f}%)")
print(f"  - 높음 (>6): {df_features['medication_high'].sum():,}건 ({df_features['medication_high'].mean()*100:.1f}%)")
print(f"  - 다약제 (≥5): {df_features['polypharmacy'].sum():,}건 ({df_features['polypharmacy'].mean()*100:.1f}%)")

# 재입원율
print(f"\nGDMT 준수 여부별 재입원율:")
for gdmt_type, col in [('Triple Therapy', 'gdmt_triple'), 
                       ('Dual Therapy', 'gdmt_dual'),
                       ('다약제', 'polypharmacy')]:
    for value, label in [(1, '예'), (0, '아니오')]:
        mask = df_features[col] == value
        if mask.sum() > 0:
            readmit_rate = df_features[mask]['readmission_30d'].mean() * 100
            n = mask.sum()
            print(f"  - {gdmt_type} {label}: {readmit_rate:.2f}% (n={n:,})")

In [ ]:
save_checkpoint(df_features, 'part4')

In [ ]:
del medication_features
del df_medications
gc.collect()

**procedures_icd table**

In [ ]:
# 시술 피처 추출
# Procedures ICD 로드
print("\n[1] Procedures ICD 로드 중...")
print("-" * 50)

df_procedures = pd.read_csv(DATA_PATH / 'procedures_icd.csv')

print(f"✓ 전체 시술: {len(df_procedures):,}건")

# 코호트에 속한 입원만 필터링
df_procedures_cohort = df_procedures[df_procedures['hadm_id'].isin(cohort_hadm_ids)].copy()

print(f"✓ 코호트 시술: {len(df_procedures_cohort):,}건")
print(f"  - 고유 hadm_id: {df_procedures_cohort['hadm_id'].nunique():,}개")
print(f"  - 고유 ICD 코드: {df_procedures_cohort['icd_code'].nunique():,}개")

# 메모리 절약
del df_procedures
gc.collect()

print(f"\n샘플 데이터:")
print(df_procedures_cohort.head())

In [ ]:
# 주요 시술 ICD 코드 정의
# 심부전 재입원과 관련된 주요 시술들
procedures_dict = {
    # 심장 시술
    'cardiac_catheterization': {
        'name': '심장도관술',
        'icd10': ['0270', '027'],  # Cardiac catheterization
        'icd9': ['3721', '3722', '3723']
    },
    'pci': {
        'name': '경피적관상동맥중재술(PCI)',
        'icd10': ['027', '0270'],  # PCI
        'icd9': ['0066', '3601', '3602', '3605', '3606', '3607']
    },
    'cabg': {
        'name': '관상동맥우회술(CABG)',
        'icd10': ['0210', '0211'],
        'icd9': ['3610', '3611', '3612', '3613', '3614', '3615', '3616', '3617', '3619']
    },
    'cardiac_device': {
        'name': '심장기기삽입(ICD/CRT)',
        'icd10': ['02H', '0JH'],
        'icd9': ['0050', '0051', '0052', '0053', '0054', '3741', '3794', '3795', '3796']
    },
    'valve_surgery': {
        'name': '판막수술',
        'icd10': ['02R', '02U'],
        'icd9': ['3521', '3522', '3523', '3524', '3525', '3526', '3527', '3528']
    },
    
    # 투석 (신기능 악화)
    'hemodialysis': {
        'name': '혈액투석',
        'icd10': ['5A1D'],
        'icd9': ['3995']
    },
    
    # 호흡 지원
    'mechanical_ventilation': {
        'name': '기계환기',
        'icd10': ['5A09', '5A19'],
        'icd9': ['9670', '9671', '9672']
    },
    'noninvasive_ventilation': {
        'name': '비침습환기',
        'icd10': ['5A09'],
        'icd9': ['9390']
    },
    
    # 혈관 접근
    'central_line': {
        'name': '중심정맥관',
        'icd10': ['05H', '02H'],
        'icd9': ['3893']
    },
    
    # 기타
    'intubation': {
        'name': '기관삽관',
        'icd10': ['0BH1'],
        'icd9': ['9604']
    },
    'cardioversion': {
        'name': '제세동',
        'icd10': ['5A22'],
        'icd9': ['9961', '9962']
    },
    'pacemaker_temp': {
        'name': '임시심박조율기',
        'icd10': ['02H'],
        'icd9': ['3706', '3707']
    }
}

print(f"✓ 정의된 시술 카테고리: {len(procedures_dict)}개")
for key, value in procedures_dict.items():
    print(f"  - {value['name']} ({key})")

In [ ]:
# 시술 피처 생성
# 각 hadm_id별로 시술 확인
procedure_features = {}

for hadm_id in cohort_hadm_ids:
    procedure_features[hadm_id] = {}

print("각 시술별 환자 수 계산 중...")

for proc_key, proc_info in procedures_dict.items():
    print(f"  처리 중: {proc_info['name']}...", end='')
    
    # ICD-10 매칭
    mask_icd10 = df_procedures_cohort['icd_version'] == 10
    mask_codes_10 = df_procedures_cohort['icd_code'].str.startswith(
        tuple(proc_info['icd10']), na=False
    )
    
    # ICD-9 매칭
    mask_icd9 = df_procedures_cohort['icd_version'] == 9
    mask_codes_9 = df_procedures_cohort['icd_code'].str.startswith(
        tuple(proc_info['icd9']), na=False
    )
    
    # 해당 시술이 있는 hadm_id
    mask_total = (mask_icd10 & mask_codes_10) | (mask_icd9 & mask_codes_9)
    hadm_with_proc = set(df_procedures_cohort[mask_total]['hadm_id'].unique())
    
    # 각 hadm_id에 피처 할당
    for hadm_id in cohort_hadm_ids:
        procedure_features[hadm_id][f'proc_{proc_key}'] = int(hadm_id in hadm_with_proc)
    
    print(f" {len(hadm_with_proc):,}건")

print("\n✓ 시술 피처 생성 완료")

In [ ]:
# 시술 데이터프레임 생성
# 딕셔너리를 데이터프레임으로 변환
df_proc_features = pd.DataFrame.from_dict(procedure_features, orient='index')
df_proc_features.reset_index(inplace=True)
df_proc_features.rename(columns={'index': 'hadm_id'}, inplace=True)

print(f"✓ 시술 데이터프레임 생성: {len(df_proc_features):,}건")
print(f"  - 컬럼 수: {len(df_proc_features.columns)}개")

# 메인 피처 데이터프레임과 병합
df_features = df_features.merge(
    df_proc_features,
    on='hadm_id',
    how='left'
)

print(f"\n✓ 병합 완료")
print(f"  - 최종 행 수: {len(df_features):,}건")
print(f"  - 최종 열 수: {len(df_features.columns)}개")

# 시술 통계
print("\n시술 시행률:")
for proc_key, proc_info in procedures_dict.items():
    col_name = f'proc_{proc_key}'
    if col_name in df_features.columns:
        count = df_features[col_name].sum()
        pct = count / len(df_features) * 100
        print(f"  - {proc_info['name']:<25}: {count:>6,}건 ({pct:>5.1f}%)")

In [ ]:
# 시술 개수 및 카테고리 피처 생성
# 시술 개수
procedure_cols = [col for col in df_features.columns if col.startswith('proc_')]
df_features['procedure_count'] = df_features[procedure_cols].sum(axis=1)

# 시술 개수 범주화
df_features['procedure_none'] = (df_features['procedure_count'] == 0).astype(int)
df_features['procedure_few'] = ((df_features['procedure_count'] >= 1) & 
                                 (df_features['procedure_count'] <= 2)).astype(int)
df_features['procedure_many'] = (df_features['procedure_count'] >= 3).astype(int)

# 주요 시술 카테고리
# 침습적 심장 시술
df_features['cardiac_intervention'] = (
    (df_features['proc_pci'] == 1) |
    (df_features['proc_cabg'] == 1) |
    (df_features['proc_valve_surgery'] == 1)
).astype(int)

# 중증 호흡 지원
df_features['respiratory_support'] = (
    (df_features['proc_mechanical_ventilation'] == 1) |
    (df_features['proc_intubation'] == 1)
).astype(int)

# 신대체요법
df_features['renal_replacement'] = df_features['proc_hemodialysis']

# 심장 기기
df_features['has_cardiac_device'] = df_features['proc_cardiac_device']

print("✓ 파생 피처 생성 완료")

print(f"\n시술 개수 분포:")
print(df_features['procedure_count'].value_counts().sort_index().head(10))

print(f"\n시술 부담 분포:")
print(f"  - 없음 (0): {df_features['procedure_none'].sum():,}건 ({df_features['procedure_none'].mean()*100:.1f}%)")
print(f"  - 적음 (1-2): {df_features['procedure_few'].sum():,}건 ({df_features['procedure_few'].mean()*100:.1f}%)")
print(f"  - 많음 (≥3): {df_features['procedure_many'].sum():,}건 ({df_features['procedure_many'].mean()*100:.1f}%)")

print(f"\n주요 시술 카테고리:")
print(f"  - 심장 중재술: {df_features['cardiac_intervention'].sum():,}건 ({df_features['cardiac_intervention'].mean()*100:.1f}%)")
print(f"  - 호흡 지원: {df_features['respiratory_support'].sum():,}건 ({df_features['respiratory_support'].mean()*100:.1f}%)")
print(f"  - 투석: {df_features['renal_replacement'].sum():,}건 ({df_features['renal_replacement'].mean()*100:.1f}%)")
print(f"  - 심장 기기: {df_features['has_cardiac_device'].sum():,}건 ({df_features['has_cardiac_device'].mean()*100:.1f}%)")

# 재입원율과의 관계
print(f"\n주요 시술별 재입원율:")
for proc_type, col in [('심장 중재술', 'cardiac_intervention'), 
                       ('호흡 지원', 'respiratory_support'),
                       ('투석', 'renal_replacement')]:
    for value, label in [(1, '예'), (0, '아니오')]:
        mask = df_features[col] == value
        if mask.sum() > 0:
            readmit_rate = df_features[mask]['readmission_30d'].mean() * 100
            n = mask.sum()
            print(f"  - {proc_type} {label}: {readmit_rate:.2f}% (n={n:,})")

In [ ]:
# 시술 피처 요약")
# 생성된 피처 목록
procedure_base_features = [col for col in df_features.columns if col.startswith('proc_')]
procedure_derived_features = [
    'procedure_count',
    'procedure_none', 'procedure_few', 'procedure_many',
    'cardiac_intervention', 'respiratory_support', 
    'renal_replacement', 'has_cardiac_device'
]

all_procedure_features = procedure_base_features + procedure_derived_features

print(f"\n생성된 시술 피처 수: {len(all_procedure_features)}개")
print(f"  - 개별 시술: {len(procedure_base_features)}개")
print(f"  - 파생 피처: {len(procedure_derived_features)}개")

print(f"\n현재 데이터프레임 크기:")
print(f"  - 행: {len(df_features):,}건")
print(f"  - 열: {len(df_features.columns)}개")

In [ ]:
save_checkpoint(df_features, 'part5')

In [ ]:
del df_procedures_cohort
del procedure_features
del df_proc_features
gc.collect()

icustay table + transfers table

In [ ]:
# ICU 및 입원 이력 피처 추출
#ICU 및 Transfers 데이터 로드

# ICU Stays 로드
if 'df_icustays' not in locals():
    df_icustays = pd.read_csv(DATA_PATH / 'icustays.csv')
    date_cols_icu = ['intime', 'outtime']
    for col in date_cols_icu:
        if col in df_icustays.columns:
            df_icustays[col] = pd.to_datetime(df_icustays[col])

print(f"✓ ICU Stays: {len(df_icustays):,}건")

# Transfers 로드
df_transfers = pd.read_csv(DATA_PATH / 'transfers.csv')
date_cols_transfer = ['intime', 'outtime']
for col in date_cols_transfer:
    if col in df_transfers.columns:
        df_transfers[col] = pd.to_datetime(df_transfers[col])

print(f"✓ Transfers: {len(df_transfers):,}건")

# Admissions 로드
if 'df_admissions' not in locals():
    df_admissions = pd.read_csv(DATA_PATH / 'admissions.csv')
    date_cols_adm = ['admittime', 'dischtime', 'deathtime', 'edregtime', 'edouttime']
    for col in date_cols_adm:
        if col in df_admissions.columns:
            df_admissions[col] = pd.to_datetime(df_admissions[col])

print(f"✓ Admissions: {len(df_admissions):,}건")

In [ ]:
# 현재 입원의 ICU 피처 생성
# 코호트의 각 입원에 대해 ICU 정보 추출
icu_features = {}

for hadm_id in cohort_hadm_ids:
    # 해당 입원의 ICU stay 찾기
    icu_stays = df_icustays[df_icustays['hadm_id'] == hadm_id]
    
    if len(icu_stays) > 0:
        # ICU 입실 여부
        icu_features[hadm_id] = {
            'icu_admission': 1,
            'icu_count': len(icu_stays),  # ICU 입실 횟수
            'icu_total_los': (icu_stays['outtime'] - icu_stays['intime']).dt.total_seconds().sum() / (24 * 3600)  # 총 일수
        }
    else:
        icu_features[hadm_id] = {
            'icu_admission': 0,
            'icu_count': 0,
            'icu_total_los': 0
        }

# 데이터프레임으로 변환
df_icu_features = pd.DataFrame.from_dict(icu_features, orient='index')
df_icu_features.reset_index(inplace=True)
df_icu_features.rename(columns={'index': 'hadm_id'}, inplace=True)

print(f"✓ ICU 피처 생성 완료: {len(df_icu_features):,}건")

# ICU 통계
print(f"\nICU 입실 통계:")
print(f"  - ICU 입실: {df_icu_features['icu_admission'].sum():,}건 ({df_icu_features['icu_admission'].mean()*100:.1f}%)")
print(f"  - 평균 ICU 입실 횟수: {df_icu_features[df_icu_features['icu_admission']==1]['icu_count'].mean():.2f}회")
print(f"  - 평균 ICU 체류 기간: {df_icu_features[df_icu_features['icu_admission']==1]['icu_total_los'].mean():.2f}일")

# ICU LOS 범주화
df_icu_features['icu_los_short'] = ((df_icu_features['icu_total_los'] > 0) & 
                                     (df_icu_features['icu_total_los'] <= 3)).astype(int)
df_icu_features['icu_los_long'] = (df_icu_features['icu_total_los'] > 3).astype(int)

print(f"\nICU 체류 기간 분포:")
print(f"  - ICU 없음: {(df_icu_features['icu_admission']==0).sum():,}건 ({(df_icu_features['icu_admission']==0).mean()*100:.1f}%)")
print(f"  - ICU 단기 (≤3일): {df_icu_features['icu_los_short'].sum():,}건 ({df_icu_features['icu_los_short'].mean()*100:.1f}%)")
print(f"  - ICU 장기 (>3일): {df_icu_features['icu_los_long'].sum():,}건 ({df_icu_features['icu_los_long'].mean()*100:.1f}%)")

In [ ]:
# 과거 입원 이력 피처 생성
# 각 환자의 과거 입원 이력 계산
history_features = {}

print("환자별 과거 입원 이력 계산 중...")

for idx, row in df_features.iterrows():
    subject_id = row['subject_id']
    hadm_id = row['hadm_id']
    current_admit = row['admittime']
    
    if idx % 10000 == 0:
        print(f"  처리 중: {idx:,}/{len(df_features):,}건", end='\r')
    
    # 해당 환자의 모든 입원 기록
    patient_admissions = df_admissions[df_admissions['subject_id'] == subject_id]
    
    # 현재 입원 이전의 입원만 (과거 이력)
    past_admissions = patient_admissions[patient_admissions['admittime'] < current_admit]
    
    if len(past_admissions) > 0:
        # 1. 과거 입원 횟수
        past_admission_count = len(past_admissions)
        
        # 2. 6개월 이내 입원 횟수
        six_months_ago = current_admit - pd.Timedelta(days=180)
        past_6m_count = len(past_admissions[past_admissions['admittime'] >= six_months_ago])
        
        # 3. 1년 이내 입원 횟수
        one_year_ago = current_admit - pd.Timedelta(days=365)
        past_1y_count = len(past_admissions[past_admissions['admittime'] >= one_year_ago])
        
        # 4. 가장 최근 입원으로부터의 일수
        most_recent_discharge = past_admissions['dischtime'].max()
        if pd.notna(most_recent_discharge):
            days_since_last_admission = (current_admit - most_recent_discharge).total_seconds() / (24 * 3600)
        else:
            days_since_last_admission = 999  # 매우 큰 값
        
        # 5. 과거 재입원 경험 (30일 이내)
        had_readmission = 0
        for i in range(len(past_admissions) - 1):
            admit_i = past_admissions.iloc[i]['dischtime']
            admit_next = past_admissions.iloc[i+1]['admittime']
            if pd.notna(admit_i) and pd.notna(admit_next):
                days_between = (admit_next - admit_i).total_seconds() / (24 * 3600)
                if 0 < days_between <= 30:
                    had_readmission = 1
                    break
        
    else:
        # 과거 입원 없음
        past_admission_count = 0
        past_6m_count = 0
        past_1y_count = 0
        days_since_last_admission = 999
        had_readmission = 0
    
    history_features[hadm_id] = {
        'past_admission_count': past_admission_count,
        'past_admission_6m': past_6m_count,
        'past_admission_1y': past_1y_count,
        'days_since_last_admission': days_since_last_admission,
        'had_past_readmission': had_readmission
    }

print(f"\n✓ 과거 입원 이력 피처 생성 완료: {len(history_features):,}건")

# 데이터프레임으로 변환
df_history = pd.DataFrame.from_dict(history_features, orient='index')
df_history.reset_index(inplace=True)
df_history.rename(columns={'index': 'hadm_id'}, inplace=True)

print(f"\n과거 입원 이력 통계:")
print(f"  - 과거 입원 있음: {(df_history['past_admission_count'] > 0).sum():,}건 ({(df_history['past_admission_count'] > 0).mean()*100:.1f}%)")
print(f"  - 평균 과거 입원 횟수: {df_history['past_admission_count'].mean():.2f}회")
print(f"  - 평균 6개월 이내 입원: {df_history['past_admission_6m'].mean():.2f}회")
print(f"  - 평균 1년 이내 입원: {df_history['past_admission_1y'].mean():.2f}회")
print(f"  - 과거 재입원 경험: {df_history['had_past_readmission'].sum():,}건 ({df_history['had_past_readmission'].mean()*100:.1f}%)")

In [ ]:
# 과거 입원 횟수 범주화
df_history['first_admission'] = (df_history['past_admission_count'] == 0).astype(int)
df_history['frequent_flyer'] = (df_history['past_admission_1y'] >= 3).astype(int)  # 1년에 3회 이상
df_history['recent_admission'] = (df_history['days_since_last_admission'] <= 30).astype(int)  # 30일 이내

# 6개월 입원 범주
df_history['past_6m_none'] = (df_history['past_admission_6m'] == 0).astype(int)
df_history['past_6m_few'] = ((df_history['past_admission_6m'] >= 1) & 
                              (df_history['past_admission_6m'] <= 2)).astype(int)
df_history['past_6m_many'] = (df_history['past_admission_6m'] >= 3).astype(int)

print("✓ 범주화 완료")

print(f"\n입원 패턴:")
print(f"  - 첫 입원: {df_history['first_admission'].sum():,}건 ({df_history['first_admission'].mean()*100:.1f}%)")
print(f"  - Frequent Flyer (1년≥3회): {df_history['frequent_flyer'].sum():,}건 ({df_history['frequent_flyer'].mean()*100:.1f}%)")
print(f"  - 최근 입원 (≤30일): {df_history['recent_admission'].sum():,}건 ({df_history['recent_admission'].mean()*100:.1f}%)")

print(f"\n6개월 입원 빈도:")
print(f"  - 없음 (0): {df_history['past_6m_none'].sum():,}건 ({df_history['past_6m_none'].mean()*100:.1f}%)")
print(f"  - 적음 (1-2): {df_history['past_6m_few'].sum():,}건 ({df_history['past_6m_few'].mean()*100:.1f}%)")
print(f"  - 많음 (≥3): {df_history['past_6m_many'].sum():,}건 ({df_history['past_6m_many'].mean()*100:.1f}%)")

In [ ]:
# ICU 피처 병합
df_features = df_features.merge(df_icu_features, on='hadm_id', how='left')
print(f"✓ ICU 피처 병합 완료")

# 입원 이력 피처 병합
df_features = df_features.merge(df_history, on='hadm_id', how='left')
print(f"✓ 입원 이력 피처 병합 완료")

print(f"\n현재 데이터프레임 크기:")
print(f"  - 행: {len(df_features):,}건")
print(f"  - 열: {len(df_features.columns)}개")

# 재입원율 분석
print("\n" + "=" * 80)
print("ICU 및 입원 이력별 재입원율")
print("=" * 80)

print(f"\n[1] ICU 입실 여부별:")
for value, label in [(1, 'ICU 있음'), (0, 'ICU 없음')]:
    mask = df_features['icu_admission'] == value
    if mask.sum() > 0:
        readmit_rate = df_features[mask]['readmission_30d'].mean() * 100
        n = mask.sum()
        print(f"  - {label}: {readmit_rate:.2f}% (n={n:,})")

print(f"\n[2] 첫 입원 여부별:")
for value, label in [(1, '첫 입원'), (0, '과거 입원 있음')]:
    mask = df_features['first_admission'] == value
    if mask.sum() > 0:
        readmit_rate = df_features[mask]['readmission_30d'].mean() * 100
        n = mask.sum()
        print(f"  - {label}: {readmit_rate:.2f}% (n={n:,})")

print(f"\n[3] Frequent Flyer 여부별:")
for value, label in [(1, 'Frequent Flyer'), (0, '아니오')]:
    mask = df_features['frequent_flyer'] == value
    if mask.sum() > 0:
        readmit_rate = df_features[mask]['readmission_30d'].mean() * 100
        n = mask.sum()
        print(f"  - {label}: {readmit_rate:.2f}% (n={n:,})")

print(f"\n[4] 과거 재입원 경험별:")
for value, label in [(1, '과거 재입원 있음'), (0, '과거 재입원 없음')]:
    mask = df_features['had_past_readmission'] == value
    if mask.sum() > 0:
        readmit_rate = df_features[mask]['readmission_30d'].mean() * 100
        n = mask.sum()
        print(f"  - {label}: {readmit_rate:.2f}% (n={n:,})")

print(f"\n[5] 최근 입원 (30일 이내) 여부별:")
for value, label in [(1, '최근 입원'), (0, '아니오')]:
    mask = df_features['recent_admission'] == value
    if mask.sum() > 0:
        readmit_rate = df_features[mask]['readmission_30d'].mean() * 100
        n = mask.sum()
        print(f"  - {label}: {readmit_rate:.2f}% (n={n:,})")

In [ ]:
# 생성된 피처 목록
icu_features_list = ['icu_admission', 'icu_count', 'icu_total_los', 'icu_los_short', 'icu_los_long']
history_features_list = [
    'past_admission_count', 'past_admission_6m', 'past_admission_1y',
    'days_since_last_admission', 'had_past_readmission',
    'first_admission', 'frequent_flyer', 'recent_admission',
    'past_6m_none', 'past_6m_few', 'past_6m_many'
]

all_part6_features = icu_features_list + history_features_list

print(f"\n생성된 피처 수: {len(all_part6_features)}개")
print(f"  - ICU 피처: {len(icu_features_list)}개")
print(f"  - 입원 이력 피처: {len(history_features_list)}개")

print(f"\n현재 데이터프레임 크기:")
print(f"  - 행: {len(df_features):,}건")
print(f"  - 열: {len(df_features.columns)}개")

In [ ]:
save_checkpoint(df_features, 'part6')

In [ ]:
del df_icu_features
del df_history
del icu_features
del history_features
gc.collect()

피처 병함(최종본 아님)

In [ ]:
# 최종 피처 정리 및 저장
#현재 데이터 상태 확인

print(f"데이터프레임 크기:")
print(f"  - 행: {len(df_features):,}건")
print(f"  - 열: {len(df_features.columns)}개")

print(f"\n기본 정보:")
print(f"  - 재입원: {df_features['readmission_30d'].sum():,}건 ({df_features['readmission_30d'].mean()*100:.2f}%)")
print(f"  - 비재입원: {(df_features['readmission_30d']==0).sum():,}건 ({(df_features['readmission_30d']==0).mean()*100:.2f}%)")
print(f"  - 고유 환자: {df_features['subject_id'].nunique():,}명")

# 메모리 사용량
import sys
mem_mb = sys.getsizeof(df_features) / (1024 * 1024)
print(f"\n메모리 사용량: {mem_mb:.1f} MB")

In [ ]:
# 결측치 확인
missing_stats = []
for col in df_features.columns:
    missing_count = df_features[col].isnull().sum()
    if missing_count > 0:
        missing_pct = missing_count / len(df_features) * 100
        missing_stats.append({
            'column': col,
            'missing_count': missing_count,
            'missing_pct': missing_pct
        })

if len(missing_stats) > 0:
    df_missing = pd.DataFrame(missing_stats)
    df_missing = df_missing.sort_values('missing_count', ascending=False)
    
    print(f"\n⚠ 결측치가 있는 컬럼: {len(df_missing)}개")
    print("\n상위 20개:")
    for idx, row in df_missing.head(20).iterrows():
        print(f"  - {row['column']:<35}: {row['missing_count']:>6,}건 ({row['missing_pct']:>5.2f}%)")
else:
    print("\n✓ 결측치 없음!")

# 날짜 컬럼 제외하고 수치형 컬럼의 결측치만
numeric_cols = df_features.select_dtypes(include=[np.number]).columns
missing_numeric = df_features[numeric_cols].isnull().sum()
missing_numeric = missing_numeric[missing_numeric > 0]

if len(missing_numeric) > 0:
    print(f"\n수치형 컬럼 중 결측치:")
    for col, count in missing_numeric.items():
        print(f"  - {col}: {count:,}건")
else:
    print(f"\n✓ 수치형 컬럼에 결측치 없음!")

In [ ]:
# 날짜 컬럼 (dod)은 결측 허용 (사망하지 않은 환자)
# 다른 결측치가 있다면 처리

# dod 외 결측치 확인
non_date_cols = [col for col in df_features.columns if col not in ['dod']]
missing_non_date = df_features[non_date_cols].isnull().sum()
missing_non_date = missing_non_date[missing_non_date > 0]

if len(missing_non_date) > 0:
    print(f"⚠ 처리 필요한 결측치: {len(missing_non_date)}개 컬럼")
    
    for col in missing_non_date.index:
        # 수치형 컬럼: 0으로 대체 (대부분 이진형)
        if df_features[col].dtype in ['int64', 'float64']:
            df_features[col].fillna(0, inplace=True)
            print(f"  ✓ {col}: 0으로 대체")
        # 범주형: 'Unknown'으로
        else:
            df_features[col].fillna('Unknown', inplace=True)
            print(f"  ✓ {col}: 'Unknown'으로 대체")
    
    print(f"\n✓ 결측치 처리 완료")
else:
    print("✓ 처리 필요한 결측치 없음 (dod 제외)")

# 최종 확인
total_missing = df_features.drop(columns=['dod'], errors='ignore').isnull().sum().sum()
print(f"\n최종 결측치 (dod 제외): {total_missing:,}개")

In [ ]:
# 피처 타입 분류
# 컬럼 분류
id_cols = ['subject_id', 'hadm_id']
date_cols = ['admittime', 'dischtime', 'dod', 'admit_year']
target_col = ['readmission_30d']
auxiliary_cols = ['days_to_readmission', 'hospital_expire_flag', 
                  'admission_type', 'discharge_location']

# 피처 컬럼 (모델링에 사용)
feature_cols = [col for col in df_features.columns 
                if col not in id_cols + date_cols + target_col + auxiliary_cols]

print(f"전체 컬럼 분류:")
print(f"  - ID 컬럼: {len(id_cols)}개")
print(f"  - 날짜 컬럼: {len(date_cols)}개")
print(f"  - Target: {len(target_col)}개")
print(f"  - 보조 정보: {len(auxiliary_cols)}개")
print(f"  - 피처 (모델링용): {len(feature_cols)}개")

# 피처를 카테고리별로 분류
print(f"\n피처 카테고리별 분류:")

categories = {
    '인구학적': ['age_', 'gender_', 'admit_', 'los_'],
    '진단': ['comorb_'],
    '약물': ['med_', 'gdmt_', 'medication_', 'polypharmacy', 'on_diuretics'],
    '시술': ['proc_', 'procedure_', 'cardiac_intervention', 'respiratory_support', 
             'renal_replacement', 'has_cardiac_device'],
    'ICU': ['icu_'],
    '입원이력': ['past_', 'first_admission', 'frequent_flyer', 'recent_admission', 
                'had_past_readmission', 'days_since_last_admission']
}

category_counts = {}
for category, prefixes in categories.items():
    count = sum(1 for col in feature_cols 
                if any(col.startswith(p) or col == p for p in prefixes))
    category_counts[category] = count
    print(f"  - {category}: {count}개")

print(f"\n총 피처: {len(feature_cols)}개")

In [ ]:
# 연속형 / 범주형 피처 구분
# 연속형 피처 (값이 연속적)
continuous_features = []
# 범주형 피처 (0/1 이진형 또는 몇 개 값만)
categorical_features = []

for col in feature_cols:
    unique_count = df_features[col].nunique()
    
    if unique_count == 2:
        # 이진형
        categorical_features.append(col)
    elif unique_count <= 10 and df_features[col].dtype in ['int64', 'float64']:
        # 10개 이하 고유값 → 범주형으로 간주
        categorical_features.append(col)
    elif df_features[col].dtype in ['int64', 'float64']:
        # 수치형이고 고유값 많음 → 연속형
        continuous_features.append(col)
    else:
        # 문자열 등 → 범주형
        categorical_features.append(col)

print(f"연속형 피처: {len(continuous_features)}개")
print("  예시:")
for col in continuous_features[:10]:
    print(f"    - {col}")
if len(continuous_features) > 10:
    print(f"    ... 외 {len(continuous_features)-10}개")

print(f"\n범주형/이진형 피처: {len(categorical_features)}개")
print("  예시:")
for col in categorical_features[:10]:
    print(f"    - {col}")
if len(categorical_features) > 10:
    print(f"    ... 외 {len(categorical_features)-10}개")

In [ ]:
# 연속형 피처 기본 통계
if len(continuous_features) > 0:
    # 연속형 피처 통계
    stats_df = df_features[continuous_features].describe().T
    stats_df['missing'] = df_features[continuous_features].isnull().sum()
    stats_df['missing_pct'] = (stats_df['missing'] / len(df_features) * 100).round(2)
    
    print("\n주요 연속형 피처 통계:")
    display_cols = ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'missing_pct']
    print(stats_df[display_cols].head(15))
    
    # 파일로 저장
    stats_file = PROCESSED_PATH / 'feature_statistics.csv'
    stats_df.to_csv(stats_file)
    print(f"\n✓ 전체 통계 저장: {stats_file}")

print("\n" + "-" * 80)
print("범주형 피처 분포 (일부):")

# 주요 범주형 피처 분포
key_categorical = ['gender_male', 'admit_emergency', 'icu_admission', 
                   'first_admission', 'frequent_flyer', 'gdmt_triple']

for col in key_categorical:
    if col in df_features.columns:
        value_counts = df_features[col].value_counts()
        print(f"\n{col}:")
        for val, count in value_counts.items():
            pct = count / len(df_features) * 100
            print(f"  {val}: {count:,}건 ({pct:.1f}%)")

In [ ]:
# Target과의 상관관계 (Top 20)
# 수치형 피처만 선택
numeric_features = df_features[feature_cols].select_dtypes(include=[np.number]).columns

# Target과의 상관관계 계산
correlations = df_features[numeric_features].corrwith(df_features['readmission_30d'])
correlations = correlations.abs().sort_values(ascending=False)

print("\nTarget(재입원)과 상관관계 높은 피처 Top 20:")
print("\n" + "-" * 60)
for idx, (feature, corr) in enumerate(correlations.head(20).items(), 1):
    # 실제 상관계수 부호 확인
    actual_corr = df_features[numeric_features].corrwith(df_features['readmission_30d'])[feature]
    direction = "↑" if actual_corr > 0 else "↓"
    print(f"{idx:2d}. {feature:<40} {direction} {abs(corr):.4f}")

# 상관관계 저장
corr_file = PROCESSED_PATH / 'feature_correlations.csv'
correlations.to_csv(corr_file, header=['correlation'])
print(f"\n✓ 상관관계 저장: {corr_file}")

In [ ]:
# 최종 데이터셋
# 1. 전체 피처 저장 (백업 + 탐색용)
final_file = PROCESSED_PATH / 'final_features_complete.csv'
df_features.to_csv(final_file, index=False)

file_size = final_file.stat().st_size / (1024 * 1024)
print(f"\n✓ 전체 피처 저장: {final_file}")
print(f"  - 크기: {file_size:.1f} MB")
print(f"  - 행: {len(df_features):,}건")
print(f"  - 열: {len(df_features.columns)}개")
print(f"  - 용도: 백업, 추가 분석, 탐색")

# 2. 모델링용 데이터셋 (피처 + Target + ID)
# ID 컬럼 정의
id_cols = ['hadm_id']  # 추적용으로 hadm_id만 포함

# Target 컬럼
target_col = ['readmission_30d']

# 제외할 컬럼 정의
exclude_cols = [
    'subject_id',           # ID (hadm_id만으로 충분)
    'admittime',            # 날짜
    'dischtime',            # 날짜
    'dod',                  # 날짜
    'admit_year',           # 파생 정보
    'days_to_readmission',  # Target 정보 포함
    'hospital_expire_flag', # 제외 기준에서 이미 제거됨
    'admission_type',       # 이미 admit_emergency 등으로 피처화
    'discharge_location',   # 재입원 후 알 수 있는 정보
    'anchor_age',           # age_at_admission으로 대체
    'anchor_year',          # 계산용
    'anchor_year_group',    # 계산용
    'gender',               # gender_male, gender_female로 대체
    'age_group'             # 범주형 (필요하면 팀원이 선택)
]

# 피처 컬럼 (제외 컬럼 빼기)
feature_cols = [col for col in df_features.columns 
                if col not in exclude_cols + target_col + id_cols]

print(f"\n✓ 피처 선택 완료:")
print(f"  - 전체 컬럼: {len(df_features.columns)}개")
print(f"  - 제외 컬럼: {len(exclude_cols)}개")
print(f"  - 모델링 피처: {len(feature_cols)}개")
print(f"  - Target: {len(target_col)}개")
print(f"  - ID: {len(id_cols)}개")

# 모델링 데이터셋 생성
modeling_cols = id_cols + feature_cols + target_col
df_modeling = df_features[modeling_cols].copy()

modeling_file = PROCESSED_PATH / 'modeling_dataset.csv'
df_modeling.to_csv(modeling_file, index=False)

print(f"\n✓ 모델링용 데이터셋 저장: {modeling_file}")
print(f"  - 크기: {modeling_file.stat().st_size / (1024*1024):.1f} MB")
print(f"  - 행: {len(df_modeling):,}건")
print(f"  - 컬럼 구성:")
print(f"    • hadm_id: 1개 (추적용)")
print(f"    • 피처: {len(feature_cols)}개")
print(f"    • Target (readmission_30d): 1개")
print(f"  - 용도: 모델링 즉시 시작 가능")

# 3. 피처 목록 저장
feature_info = pd.DataFrame({
    'feature': feature_cols,
    'dtype': [str(df_features[col].dtype) for col in feature_cols],
    'unique_values': [df_features[col].nunique() for col in feature_cols],
    'missing': [df_features[col].isnull().sum() for col in feature_cols],
    'type': ['continuous' if col in continuous_features else 'categorical' 
             for col in feature_cols]
})

feature_info_file = PROCESSED_PATH / 'feature_list.csv'
feature_info.to_csv(feature_info_file, index=False)

print(f"\n✓ 피처 목록 저장: {feature_info_file}")
print(f"  - 연속형: {(feature_info['type']=='continuous').sum()}개")
print(f"  - 범주형: {(feature_info['type']=='categorical').sum()}개")

# 4. 제외된 컬럼 목록 (참고용)
excluded_info = pd.DataFrame({
    'excluded_column': exclude_cols,
    'reason': [
        'ID (hadm_id로 충분)',
        '날짜 정보',
        '날짜 정보',
        '날짜 정보 (사망일)',
        '파생 정보',
        'Target 정보 포함 (data leakage 위험)',
        '이미 제외 기준',
        '이미 피처화됨',
        '재입원 후 정보 (data leakage)',
        'age_at_admission으로 대체',
        '계산용',
        '계산용',
        'gender_male/female로 대체',
        '범주형 (필요시 complete에서 사용)'
    ]
})

excluded_file = PROCESSED_PATH / 'excluded_columns.csv'
excluded_info.to_csv(excluded_file, index=False)

print(f"\n✓ 제외 컬럼 목록 저장: {excluded_file}")
print(f"  - 팀원 참고: 왜 제외했는지 확인 가능")

# 5. 컬럼 목록 (텍스트)
columns_file = PROCESSED_PATH / 'final_columns.txt'
with open(columns_file, 'w') as f:
    f.write("=== 모델링 데이터셋 컬럼 ===\n")
    f.write(f"Total: {len(modeling_cols)} columns\n\n")
    f.write("ID:\n")
    for col in id_cols:
        f.write(f"  - {col}\n")
    f.write(f"\n피처 ({len(feature_cols)}개):\n")
    for col in feature_cols:
        f.write(f"  - {col}\n")
    f.write(f"\nTarget:\n")
    f.write(f"  - readmission_30d\n")

print(f"✓ 컬럼 목록 저장: {columns_file}")